# Hourly Cache Volume Filter

Filter hourly OHLCV caches stored in `pickle_hour_cache/` by removing symbols that contain any zero-volume entries.


**Workflow**:
1. Configure directories for source caches and filtered outputs.
2. Load each monthly pickle bundle.
3. Drop symbols whose `volume` column hits zero in that month.
4. Save the filtered data to `filter_hour_cache/` and persist the surviving symbol list per month in `available_symbol/`.

Run the cells sequentially after activating the project Python environment.


In [2]:
import sys

print(f'Python version: {sys.version}')
try:
    import pickle5  # type: ignore
    print('pickle5 available (protocol ≥ 5 supported).')
except ImportError:
    print('pickle5 not installed. Install with `%pip install pickle5` if protocol errors occur.')


Python version: 3.9.12 (main, Jun  1 2022, 11:38:51) 
[GCC 7.5.0]
pickle5 not installed. Install with `%pip install pickle5` if protocol errors occur.


In [3]:
from pathlib import Path
import pandas as pd
import pickle

DATA_ROOT = Path('/home/craz/crypto/crypto-data')
SOURCE_DIR = DATA_ROOT / 'pickle_hour_cache'
FILTERED_DIR = DATA_ROOT / 'filter_hour_cache'
AVAILABLE_DIR = DATA_ROOT / 'available_symbol'

FILTERED_DIR.mkdir(parents=True, exist_ok=True)
AVAILABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Reading from: {SOURCE_DIR}')
print(f'Writing filtered frames to: {FILTERED_DIR}')
print(f'Writing symbol lists to: {AVAILABLE_DIR}')


Reading from: /home/craz/crypto/crypto-data/pickle_hour_cache
Writing filtered frames to: /home/craz/crypto/crypto-data/filter_hour_cache
Writing symbol lists to: /home/craz/crypto/crypto-data/available_symbol


In [4]:
from typing import List, Tuple

def list_symbols(frame: pd.DataFrame) -> pd.Index:
    if 'symbol' in frame.columns:
        return pd.Index(frame['symbol'].unique())
    if frame.index.names and 'symbol' in frame.index.names:
        return frame.index.get_level_values('symbol').unique()
    raise KeyError("Could not find a 'symbol' column or index level.")

def load_hourly_bundle(path: Path) -> pd.DataFrame:
    try:
        frame = pd.read_pickle(path)
    except Exception as exc:
        message = str(exc).lower()
        if 'unsupported pickle protocol' in message:
            try:
                import pickle5  # type: ignore
            except ImportError as import_error:
                raise RuntimeError("pickle protocol ≥ 5 detected. Install pickle5 via `%pip install pickle5`.") from import_error
            with open(path, 'rb') as fh:
                frame = pickle5.load(fh)
        else:
            raise
    if isinstance(frame, dict):
        frame = pd.DataFrame(frame)
    return frame

def filter_zero_volume_symbols(frame: pd.DataFrame) -> Tuple[pd.DataFrame, List[str], List[str]]:
    if 'volume' not in frame.columns:
        raise KeyError("Expected a 'volume' column in the hourly bundle.")
    all_symbols = list_symbols(frame)
    zero_mask = frame['volume'] == 0

    if 'symbol' in frame.columns:
        zero_symbols = frame.loc[zero_mask, 'symbol'].unique()
        filtered = frame.loc[~frame['symbol'].isin(zero_symbols)].copy()
    else:
        symbol_index = frame.index.get_level_values('symbol')
        zero_symbols = symbol_index[zero_mask].unique()
        keep_mask = ~symbol_index.isin(zero_symbols)
        filtered = frame.loc[keep_mask].copy()

    kept_symbols = sorted(set(all_symbols) - set(zero_symbols))
    removed_symbols = sorted(set(zero_symbols))

    return filtered, kept_symbols, removed_symbols


In [5]:
summary_rows = []
source_files = sorted(SOURCE_DIR.glob('*.pkl'))

if not source_files:
    raise FileNotFoundError(f'No pickle files found in {SOURCE_DIR}.')

for source_path in source_files:
    month_key = source_path.stem.split('_')[-1]
    frame = load_hourly_bundle(source_path)
    filtered_frame, kept_symbols, removed_symbols = filter_zero_volume_symbols(frame)

    filtered_path = FILTERED_DIR / source_path.name
    filtered_frame.to_pickle(filtered_path, protocol=pickle.HIGHEST_PROTOCOL)

    symbols_path = AVAILABLE_DIR / f'{month_key}_symbols.pkl'
    with open(symbols_path, 'wb') as fh:
        pickle.dump(kept_symbols, fh, protocol=pickle.HIGHEST_PROTOCOL)

    all_symbols = list_symbols(frame)
    summary_rows.append({
        'month': month_key,
        'source_symbols': len(all_symbols),
        'kept_symbols': len(kept_symbols),
        'removed_symbols': len(removed_symbols),
        'source_rows': len(frame),
        'filtered_rows': len(filtered_frame),
        'filtered_path': str(filtered_path),
        'symbol_list_path': str(symbols_path)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,month,source_symbols,kept_symbols,removed_symbols,source_rows,filtered_rows,filtered_path,symbol_list_path
0,2020-01,10,10,0,4760,4760,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
1,2020-02,23,23,0,12676,12676,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
2,2020-03,23,23,0,17112,17112,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
3,2020-04,23,23,0,16560,16560,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
4,2020-05,24,24,0,17216,17216,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
...,...,...,...,...,...,...,...,...
65,2025-06,480,441,39,339533,311453,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
66,2025-07,503,463,40,364406,334646,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
67,2025-08,522,480,42,380859,349611,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
68,2025-09,554,505,49,386028,350748,/home/craz/crypto/crypto-data/filter_hour_cach...,/home/craz/crypto/crypto-data/available_symbol...
